<hr style="height:0px; visibility:hidden;" />

<h1><center><b>GL4U: RNAseq<b></center></h1>

# Pipeline for Processing GeneLab RNA-sequencing Data

> **The GL4U RNAseq Jupyter Notebooks (JNs) are designed to teach students how to process RNA sequencing data using the [GeneLab standard pipeline](https://github.com/nasa/GeneLab_Data_Processing/blob/master/RNAseq/README.md). Below are step-by-step instructions for processing samples derived from the soleus (aka "anti-gravity") muscle of mice that were either flown in space aboard the International Space Station (ISS) (spaceflight, FLT) or kept in an environmental simulator on Earth to serve as ground control (GC) animals during NASA's Rodent Research - 1 mission. More information about the samples analyzed here can be found in the [Open Science Data Repository](https://osdr.nasa.gov/bio/repo) under [OSD-104](https://osdr.nasa.gov/bio/repo/data/studies/OSD-104).**  

---

### RNAseq Pipeline Overview
> The RNAseq Processing JNs, parts 1 and 2, will cover the pipeline steps outlined in red. For more information about how GeneLab processes bulk RNAseq data, visit the [GeneLab Data Processing GitHub repository](https://github.com/nasa/GeneLab_Data_Processing/tree/master/RNAseq).  

<br>

<div align="center">
<img src="https://raw.githubusercontent.com/nasa/GeneLab-Training/refs/heads/GL4U_RNAseq_2024_Colab/images/rnaseq-processing-pipeline.png" alt="RNAseq processing pipeline">
</div>

<br>

<hr style="height:0px; visibility:hidden;" />
    
<h1><center>1. RNAseq processing: raw to trimmed QC</center></h1>

<div class="alert alert-block alert-success">
Here we are going to setup a directory structure to store the output data we'll generate, then we will follow the steps in the GeneLab standard RNAseq processing pipeline to generate trimmed sequence data and run quality control (QC) metrics.
</div>

---

This is notebook 1 of 3 of GL4U's RNAseq Module Set. It is expected that GL4U's Introduction Module Set has been completed already.

---

## Set up the notebook environment
Run the following commands to mount your google drive and copy all required files to your google drive folder.
> *For step-by-step instructions to mount your google drive, open the following link in a new browser tab: [How to Mount Google Drive](https://github.com/nasa/GeneLab-Training/blob/GL4U_Intro_2024_Colab/On-Demand/Mount_Google_Drive.md).*



In [1]:
# mount google drive
from google.colab import drive
drive.flush_and_unmount()
drive.mount("mnt")

Drive not mounted, so nothing to flush and unmount.
Mounted at mnt


In [2]:
# create GL4U RNAseq directory in Google Drive
import os
GL4U_RNASEQ_DIR="/content/mnt/MyDrive/NASA/GL4U/RNAseq"
if not os.path.exists(GL4U_RNASEQ_DIR):
  !mkdir -p {GL4U_RNASEQ_DIR}

In [3]:
# download OSD-104 input files from GitHub
if os.path.exists(f"{GL4U_RNASEQ_DIR}"):
    !git clone --depth 1 --branch GL4U_RNAseq_2024 https://github.com/nasa/GeneLab-Training.git /tmp/GeneLab-Training
    !mv /tmp/GeneLab-Training/OSD-104 "{GL4U_RNASEQ_DIR}"
    !mv /tmp/GeneLab-Training/download_file_urls.py "{GL4U_RNASEQ_DIR}"
    !mv /tmp/GeneLab-Training/gtf_mapper.py "{GL4U_RNASEQ_DIR}"
    !rm -rf /tmp/GeneLab-Training

Cloning into '/tmp/GeneLab-Training'...
remote: Enumerating objects: 160, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 160 (delta 41), reused 145 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (160/160), 104.56 MiB | 18.42 MiB/s, done.
Resolving deltas: 100% (41/41), done.
Updating files: 100% (135/135), done.
mv: inter-device move failed: '/tmp/GeneLab-Training/OSD-104' to '/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104'; unable to remove target: Directory not empty


---
<br>
<div class="alert alert-block alert-info">
<b>Note:</b> Google Colab does not currently offer the bash programming language. However, we are able to execute bash commands in the python programming language by adding characters such as a "!" or a "%" in front of the command as you will see throughout this JN.
</div>

<a id="setup"></a>

## 0. Setup

We'll start by moving to the location that contains the files we will use in this JN:

In [4]:
%cd /content/mnt/MyDrive/NASA/GL4U/RNAseq/
!pwd

/content/mnt/MyDrive/NASA/GL4U/RNAseq
/content/mnt/MyDrive/NASA/GL4U/RNAseq


**Let's see the directory and all sub-directories that we will be using:**
> During RNAseq processing, you will generate several, likely 100s, of files so it is important to keep the quality control (QC) and processed data files organized. Below, you'll see the directory structure we use at GeneLab to keep our files organized.

<div class="alert alert-block alert-info">
<b>Note:</b><br>

*For the purposes of this bootcamp we are only using a subset of the [OSD-104](https://osdr.nasa.gov/bio/repo/data/studies/OSD-104) RNAseq raw reads from the spaceflight (FLT) and respective ground control (GC) soleus muscle samples prepared using the ribo-depletion library preparation method.*

</div>

In [5]:
!find ./OSD-104 -maxdepth 3 -type d | sort

./OSD-104
./OSD-104/00-RawData
./OSD-104/00-RawData/Fastq
./OSD-104/00-RawData/FastQC_Reports
./OSD-104/00-RawData/FastQC_Reports/raw_multiqc_report
./OSD-104/01-TG_Preproc
./OSD-104/01-TG_Preproc/Fastq
./OSD-104/01-TG_Preproc/FastQC_Reports
./OSD-104/01-TG_Preproc/Trimming_Reports
./OSD-104/02-STAR_Alignment
./OSD-104/02-STAR_Alignment/FLT_M23
./OSD-104/02-STAR_Alignment/FLT_M23/FLT_M23__STARgenome
./OSD-104/02-STAR_Alignment/FLT_M23/FLT_M23__STARpass1
./OSD-104/02-STAR_Alignment/Log_files
./OSD-104/03-RSEM_Counts
./OSD-104/03-RSEM_Counts/FLT_M23.stat
./OSD-104/03-RSEM_Counts/Log_files
./OSD-104/Metadata
./OSD-104/processing_scripts
./OSD-104/processing_scripts/02-STAR_Alignment
./OSD-104/processing_scripts/02-STAR_Alignment/STAR_align_out_logs
./OSD-104/References
./OSD-104/RSeQC_analyses
./OSD-104/RSeQC_analyses/infer_exp


<div class="alert alert-block alert-info">
<b><i>Code Breakdown</i></b>
<br>
    
- `find`   - the primary command we're using, which can be used to look for directories and files
    - `./OSD-104`   - this is a positional argument that tells `find` to search in the "OSD-104" folder that is in our current working directory
    - `-type d`   - this specifies what we're looking for, the d indicates "directories"
    - `-maxdepth 3` - this tells find to only list directories up to 3 levels deep
- `| sort` - directs the output from the find command into the sort command using a pipe
  
</div>

<a class="anchor" id="paths"></a>

### Define Paths to GL4U: RNAseq Bootcamp Processed Data

To save time and conserve resources, some of the processing steps below will utilize processed data that have already been generated for this course and made available in the directory structure we observed with the command above. Run the following commands to set up variables that define the paths to those files:

In [6]:
metadata="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/Metadata"
raw_fastq="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/00-RawData/Fastq"
raw_fastqc="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/00-RawData/FastQC_Reports"
raw_multiqc="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/00-RawData/FastQC_Reports/raw_multiqc_report"
trimmed_fastq="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/01-TG_Preproc/Fastq"
trimmed_fastqc="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/01-TG_Preproc/FastQC_Reports"
trimmed_multiqc="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/01-TG_Preproc/FastQC_Reports/trimmed_multiqc_report"
STAR_output="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/02-STAR_Alignment"
STAR_logs="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/02-STAR_Alignment/Log_files"
aligned_multiqc="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/02-STAR_Alignment/aligned_multiqc_report"
REFs="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/References"
RSeQC_infer_exp="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/RSeQC_analyses/infer_exp"
RSeQC_infer_exp_multiqc="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/RSeQC_analyses/infer_exp/infer_exp_multiqc_report"
RSEM_output="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/03-RSEM_Counts"
RSEM_logs="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/03-RSEM_Counts/Log_files"
count_multiqc="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/03-RSEM_Counts/count_multiqc_report"
processing="/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/processing_scripts"

There is a file with information about the samples we will be processing in the `Metadata` sub-directory. Let's use the cat command to take a look at that file:

In [7]:
!cat $metadata/OSD-104_Sample_Metadata.csv

sample,condition
FLT_M23,FLT
FLT_M24,FLT
FLT_M25,FLT
FLT_M26,FLT
FLT_M27,FLT
FLT_M28,FLT
GC_M33,GC
GC_M34,GC
GC_M35,GC
GC_M36,GC
GC_M37,GC
GC_M38,GC


The columns are not perfectly aligned across each row when using `cat`. This just has to do with how the print-out is positioning things.

We can use a command called `column` to help with formatting the print-out (where `-s,` specifies the comma as the delimiter and `-t` tells it to try to make a table):

In [8]:
!column -s, -t $metadata/OSD-104_Sample_Metadata.csv

sample   condition
FLT_M23  FLT
FLT_M24  FLT
FLT_M25  FLT
FLT_M26  FLT
FLT_M27  FLT
FLT_M28  FLT
GC_M33   GC
GC_M34   GC
GC_M35   GC
GC_M36   GC
GC_M37   GC
GC_M38   GC


**Take a look at the `OSD-104_Sample_Metadata.csv` file above and answer the following questions:**

1. How many samples are there?

2. How many samples are derived from spaceflight (FLT)? How many from ground control (GC)? How do you know?

**Input your responses to each question in the text boxes below:**
> *Answer each question in your own words BEFORE checking your answers against the solutions below. Please do not copy and paste the answers provided.*




```
# This is formatted as code
```

Question 1: There are 12 samples in total.


Question 2: There are 6 spaceflight (FLT) samples and 6 ground control (GC) samples. I know this by looking at the 'condition' column in the metadata table, where the samples are explicitly labeled as either 'FLT' or 'GC'.


<div class="alert alert-block alert-success">

<details>
<summary><b>Solutions</b></summary>

<br>

<b>1.</b> 12 samples<br>
<b>2.</b> 6 samples from FLT and 6 samples from GC, going by what's in the "condition" column
    
</details>
</div>

**Okay, now that we're done with our housekeeping, it's time to start processing RNAseq data!**

<div class="alert alert-block alert-info">
<b>Note:</b><br>

*For the purposes of this bootcamp we will only be running one sample, FLT_M23, through each step of the pipeline.*

</div>

---

<a class="anchor" id="rawqc"></a>

## 1. Raw Data Quality Control (QC)

<a class="anchor" id="rawfastqc"></a>

### 1a. Raw Data QC with FastQC

First, we will assess the quality of the raw RNA sequence data using the [FastQC](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/) program. Short-read sequence data are stored in fastq files, which contain several (often millions) of reads. Each read consists of the following 4 line structure:

 > Line 1 - Begins with @, followed by information about the sequencing run such as the sequencing platform, run number, flow cell ID and cluster location, read number (forward/reverse), and/or the sample index.
 >
 > Line 2 - Contains the sequence, written as base calls (A, T, C, or G). Note that the sequence length is equal to the number of cycles in the sequencing run.
 >
 > Line 3 - A separator line, which begins with a plus (+) sign.
 >
 > Line 4 - Quality scores of each base call that are Phred +33 encoded and use <a href="http://drive5.com/usearch/manual/quality_score.html">ASCII characters</a> to represent the quality of the bases.

Let's take a look at the raw fastq files we will be using:

In [9]:
!ls -1 $raw_fastq/

FLT_M23_R1_raw.fastq.gz
FLT_M23_R2_raw.fastq.gz


Now, let's look at the first couple reads of the forward (R1) fastq file for sample FLT_M23:
> *Note: The `zcat` command is similar to the `cat` command we learned in the Unix intro, but it will uncompress gzipped files first then print the contents.*

In [10]:
!zcat $raw_fastq/FLT_M23_R1_raw.fastq.gz | head -n 8

@J00113:162:H7W32BBXX:1:1101:11261:1947 1:N:0:GGAGAA
GGCATGCAAAGATTAAAGTAGTGAAATAAAAAATAAATGGCTCTACTTTGGGCAAAGAAAACCATCTTTATGAAGAAGAAATTTAAATGCTGGATCAAAC
+
AAFFFJJFFJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJ-77<F7<<FJJFFFJJJJJJJJAFF<<FFFJJJJJFJJJJJJJJJJ<JJJJFAJFFJJF
@J00113:162:H7W32BBXX:1:1101:21389:1947 1:N:0:GGAGAA
CACTGGTAGAATTTGTGTAGTCCCACCTTGCATAAATTCAACTTATGAAGTATGGTGAGCTGTGTCACAAAAACAAAACAAAATTGAAAAAATTAATGAA
+
AAFFFJJJJJJJJJJJJJJFFFJJJJJFFF7FFJFJJJ<<<F<<<<FJFJJJJJJJJFJJAFFJJFFJFJJJJFFFJFFJJJJJJJJJJJJJJJJJJJJJ


It would be difficult to determine the quality of sequence data if we had to interpret the individual quality score for each base of each read of a file. Luckily we don't have to!

FastQC will evaluate the information provided for each read in the fastq files and generate a summary report about the quality of the sequence data. Assessing the raw sequence quality is important for determining if and how the reads need to be trimmed.

Before we can run FastQC, we first need to install it by running the following commands:

In [11]:
# install FastQC
if not os.path.exists(f"{GL4U_RNASEQ_DIR}/FastQC/fastqc"):
  !mkdir {GL4U_RNASEQ_DIR}/FastQC
  !wget -O {GL4U_RNASEQ_DIR}/fastqc.zip https://www.bioinformatics.babraham.ac.uk/projects/fastqc/fastqc_v0.12.1.zip
  !unzip {GL4U_RNASEQ_DIR}/fastqc.zip -d {GL4U_RNASEQ_DIR} > /dev/null
!chmod +x {GL4U_RNASEQ_DIR}/FastQC/fastqc

# remove the fastqc.zip file as we don't need it anymore
!rm -f {GL4U_RNASEQ_DIR}/fastqc.zip

In [12]:
# check the FastQC version
!{GL4U_RNASEQ_DIR}/FastQC/fastqc --version

FastQC v0.12.1


In [13]:
# remove the fastqc.zip file as we don't need it anymore
!rm -f {GL4U_RNASEQ_DIR}/fastqc.zip

We're now ready to run FastQC on the raw reads of our FLT_M23 sample.

In [14]:
# run FastQC on the raw reads
!{GL4U_RNASEQ_DIR}/FastQC/fastqc -o $raw_fastqc $raw_fastq/*raw.fastq.gz

application/gzip
Started analysis of FLT_M23_R1_raw.fastq.gz
application/gzip
Approx 5% complete for FLT_M23_R1_raw.fastq.gz
Approx 10% complete for FLT_M23_R1_raw.fastq.gz
Approx 15% complete for FLT_M23_R1_raw.fastq.gz
Approx 20% complete for FLT_M23_R1_raw.fastq.gz
Approx 25% complete for FLT_M23_R1_raw.fastq.gz
Approx 30% complete for FLT_M23_R1_raw.fastq.gz
Approx 35% complete for FLT_M23_R1_raw.fastq.gz
Approx 40% complete for FLT_M23_R1_raw.fastq.gz
Approx 45% complete for FLT_M23_R1_raw.fastq.gz
Approx 50% complete for FLT_M23_R1_raw.fastq.gz
Approx 55% complete for FLT_M23_R1_raw.fastq.gz
Approx 60% complete for FLT_M23_R1_raw.fastq.gz
Approx 65% complete for FLT_M23_R1_raw.fastq.gz
Approx 70% complete for FLT_M23_R1_raw.fastq.gz
Approx 75% complete for FLT_M23_R1_raw.fastq.gz
Approx 80% complete for FLT_M23_R1_raw.fastq.gz
Approx 85% complete for FLT_M23_R1_raw.fastq.gz
Approx 90% complete for FLT_M23_R1_raw.fastq.gz
Approx 95% complete for FLT_M23_R1_raw.fastq.gz
Approx 100%

<div class="alert alert-block alert-info">
<b><i>Code Breakdown</i></b>

**Parameter Definitions:**

* `-o` – the output directory to store the FastQC results
* `*raw.fastq.gz` – the input reads are specified as a positional argument, and can be given all at once with wildcards, or as individual arguments with spaces in between them

**Input Data:**
- `*raw.fastq.gz` (raw reads)

**Output Data:**
- `*fastqc.html` (FastQC report: a webpage file containing graphical representations of the raw sequence quality)
- `*fastqc.zip` (FastQC data: a compressed directory containing the quality reports of all raw sequences assessed)

<br>
</div>

___________



FastQC is finished running when you see the "Analysis complete for" statement for both the forward (R1) and reverse (R2) files of our sample.

Let's list the output files that were generated:

In [15]:
!ls -1 $raw_fastqc/FLT_M23*

/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/00-RawData/FastQC_Reports/FLT_M23_R1_raw_fastqc.html
/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/00-RawData/FastQC_Reports/FLT_M23_R1_raw_fastqc.zip
/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/00-RawData/FastQC_Reports/FLT_M23_R2_raw_fastqc.html
/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/00-RawData/FastQC_Reports/FLT_M23_R2_raw_fastqc.zip


Open the raw data `FastQC_Reports` directory in the side panel of this Colab notebook by clicking: `/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/00-RawData/FastQC_Reports`, as shown below.
> *Note: These files are stored in your Google Drive folder.*

<br>

<div align="center">
<img src="https://raw.githubusercontent.com/nasa/GeneLab-Training/refs/heads/GL4U_RNAseq_2024_Colab/images/gd_raw_fastqc_dir.png" alt="Open Raw FastQC Reports directory">
</div>
<br>

Click the 3 dots next to the `FLT_M23_R1_raw_fastqc.html` and `FLT_M23_R2_raw_fastqc.html` files in the left side panel to download them to your computer as shown below:

<br>

<div align="center">
<img src="https://raw.githubusercontent.com/nasa/GeneLab-Training/refs/heads/GL4U_RNAseq_2024_Colab/images/gd_dl_raw_fastqc.png" alt="Download fastqc HTML files">
</div>
<br>

Navigate to the HTML files you downloaded on your computer then double click on them to open in your web browser to view the FastQC reports.

<div class="alert alert-block alert-info">
<b>Note:</b><br>

*As we covered in the GL4U Intro course, there are many modules included in the `FastQC` program, with the documentation for each linked on [this page](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/Help/3%20Analysis%20Modules/). While these are summarized in a pass (green check), warning (yellow exclamation point), and fail (red X) fashion, it is important to remember that those indicators are based on expecting the data are completely random and diverse, which is often not the case depending on the type of sequencing that was done.*

</div>

---

<a class="anchor" id="rawmultiqc"></a>

### 1b. Compile Raw Data QC with MultiQC

Rather than viewing the FastQC reports of each fastq file for all 12 samples individually, we can use the [MultiQC](https://docs.seqera.io/multiqc) program to compile the FastQC data from all samples and generate a single report (and associated data files). This allows us to view and assess the raw sequence data of all samples at once.

Before we can run MultiQC, we first need to install it by running the following commands:

In [16]:
# install MultiQC
%pip install --quiet multiqc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.3 MB/s eta 0:00:00


In [17]:
# check the MultiQC version
!multiqc --version

multiqc, version 1.33


We're now ready to compile the raw FastQC data from our 12 samples using MultiQC.

In [18]:
!multiqc --interactive -n raw_multiqc -o $raw_multiqc $raw_fastqc/


/// ]8;id=751619;https://multiqc.info\MultiQC]8;;\ 🔍 v1.33

       file_search | Search path: /content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/00-RawData/FastQC_Reports
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 47/47  
            fastqc | Found 24 reports
      exec_modules | Updating module "FastQC"
     write_results | Existing reports found, adding suffix to filenames. Use '--force' to overwrite.
     write_results | Data        : OSD-104/00-RawData/FastQC_Reports/raw_multiqc_report/raw_multiqc_data_1
     write_results | Report      : OSD-104/00-RawData/FastQC_Reports/raw_multiqc_report/raw_multiqc_1.html
           multiqc | MultiQC complete


<div class="alert alert-block alert-info">
<b><i>Code Breakdown</i></b>

**Parameter Definitions:**

* `--interactive` – force reports to use interactive plots
* `-n` – prefix name for the output files
* `-o` – specifies the output directory to store the MultiQC results
* `$raw_fastqc/` – the directory holding the raw FastQC data from each sample, provided as a positional argument

**Input Data:**
- `*fastqc.zip` (FastQC data)

**Output Data:**
- `raw_multiqc_data` (directory containing the compiled raw FastQC data from each sample, used to create the MultiQC report)
- `raw_multiqc.html` (MultiQC report: an interactive webpage file containing graphical representations of the compiled raw sequence quality)

</div>



---
MultiQC is finished running when the loading animation on the left end of the code cell stops and a green checkmark appears. In a standard bash shell environment, if the multiQC job finishes successfully, you will see a "MultiQC complete" statement at the end of the standard output, and could check that the number of reports found matches what you expect.
> Note: For paired-end data, we expect a report for both the forward (R1) and reverse (R2) reads of each sample. Since we have 12 paired-end samples, MultiQC should detect 24 FastQC reports.

Let's list the output files that were generated:

In [19]:
!ls -1 $raw_multiqc/

raw_multiqc_1.html
raw_multiqc_data
raw_multiqc_data_1
raw_multiqc.html


Open the `raw_multiqc_report` directory in the side panel of this Colab notebook by clicking: `/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/00-RawData/FastQC_Reports/raw_multiqc_report`, as shown below.
> *Note: These files are stored in your Google Drive folder.*

<br>

<div align="center">
<img src="https://raw.githubusercontent.com/nasa/GeneLab-Training/refs/heads/GL4U_RNAseq_2024_Colab/images/gd_raw_multiqc_dir.png" alt="Open the raw multiQC report directory">
</div>
<br>

Click the 3 dots next to the `raw_multiqc.html` file in the left side panel to download it to your computer as shown below.

<br>

<div align="center">
<img src="https://raw.githubusercontent.com/nasa/GeneLab-Training/refs/heads/GL4U_RNAseq_2024_Colab/images/gd_dl_raw_multiqc.png" alt="Download raw multiqc HTML file">
</div>
<br>

Navigate to the HTML file you downloaded on your computer then double click on it to open in your web browser to view the compiled FastQC reports for all 12 samples, which should look similar to the following:

<br>

<div align="center">
<img src="https://raw.githubusercontent.com/nasa/GeneLab-Training/refs/heads/GL4U_RNAseq_2024_Colab/images/gd_open_raw_multiqc.png" alt="Open raw multiQC report">
</div>
<br>
<div class="alert alert-block alert-info">
<b>Note:</b><br>
    
Click [here](https://docs.seqera.io/multiqc/modules/fastqc) for a description of the MultiQC plots derived from FastQC data.

</div>

<div class="alert alert-block alert-warning">
<a class="anchor" id="enableJS"></a>

##### Enable JavaScript
    
Be sure to click the "Trust HTML" button in the top left corner of the MultiQC report to be able to see all the plots.

If the plots are still not visible, make sure JavaScript is enabled on the web-browser you are using. Below are instructions for how to enable JavaScript on common web-browsers:
- Google Chrome:
  - Click the three-dot menu in the top right corner and go to Settings.
  - Scroll down and click Privacy and security on the left.
  - Select Site Settings.
  - Scroll down to JavaScript and make sure it's set to Allowed.
- Mozilla Firefox:
  - Type about:config in the address bar and press Enter.
  - Click the "Accept the Risk and Continue" button.
  - In the search bar, type javascript.enabled.
  - Ensure the value is set to true. If it's false, double-click to change it to true.
- Microsoft Edge:
  - Click the three-dot menu in the top right corner and go to Settings.
  - Scroll down and select Cookies and site permissions.
  - Under Site permissions, click JavaScript.
  - Make sure the toggle is set to Allowed.
- Safari (Mac):
  - Open the Safari menu and select Preferences.
  - Go to the Security tab.
  - Make sure the checkbox Enable JavaScript is checked.

</div>

**Take a look at the MultiQC report of the raw FastQC data and answer the following questions:**

1. Which sample was sequenced at the greatest read depth? the least read depth? (Hint: See *M Seqs* under "General Statistics")

2. What do you notice about the quality of the raw sequence data? Are there any differences between the forward and reverse reads?

3. Were adapters detected?

4. Should we trim and/or filter the data before aligning to the reference genome? Why or why not?

**Input your responses to each question in the text boxes below:**
> *Answer each question in your own words BEFORE checking your answers against the solutions below. Please do not copy and paste the answers provided.*


Question 1: In this specific dataset, all samples exhibit a uniform read depth of 0.1M reads.

Question 2: The overall sequence quality is high, with Phred scores generally staying above Q25. However, a systematic drop in quality is observed in the reverse reads compared to the forward reads.


Question 3: Yes, Illumina Universal Adapters were detected, particularly towards the end of the reads (3' end).


Question 4: Yes, we must perform trimming and filtering. Removing the detected adapters and filtering out low-quality bases from the ends of the reads is essential to minimize mismatches during the alignment phase. This step is non-negotiable for achieving high-confidence biological results.



<br>

<div class="alert alert-block alert-success">

<details>
<summary><b>Q1 Solution</b></summary>

<br>

All samples show the same read depth, 0.1M reads.
> *Note: This is because these reads were purposfully truncated to 100,000. Normally you'll find that samples have different read depths. In RNAseq studies, it is important for samples to be sequenced at similar depths to enable downstream gene expression comparisons.*
    
</details>
</div>


<div class="alert alert-block alert-success">

<details>
<summary><b>Q2 Solution</b></summary>

<br>

The quality is overall good (>Q25).  
    
The reverse reads have slightly lower quality than the forward reads.
> This may be due to the fact that the reverse reads are sequenced after the forward reads. Therefore the template cDNA and/or the reagents may have degraded slightly, reducing the quality of the reverse reads.
    
</details>
</div>


<div class="alert alert-block alert-success">

<details>
<summary><b>Q3 Solution</b></summary>

<br>

Yes, Illumina Universal Adapters are detected towards the end of some reads for every sample, although they make up less than 2% of the total reads for each sample.

</details>
</div>


<div class="alert alert-block alert-success">

<details>
<summary><b>Q4 Solution</b></summary>

<br>

While these data look pretty good overall, it is still a good idea to trim/filter them to remove the adapters and since these quality plots only show the averages, there may be some reads with poor quality we might still want to filter out.

</details>
</div>

---

<a class="anchor" id="trimfilter"></a>

## 2. Trim/Filter Raw Data

<a class="anchor" id="trim"></a>

### 2a. Trim and Filter Raw Sequence Data with Trim Galore

To ensure maximal sequence alignment to the reference genome, our next step is to clean up the raw sequence data using [Trim Galore](https://www.bioinformatics.babraham.ac.uk/projects/trim_galore/), which utilizes the [cutadapt](https://cutadapt.readthedocs.io/en/stable/) software. During this step, we filter the raw reads to remove any sequencing adapters detected, which were artificially added to enable sequencing of the sample RNA, as well as low-quality base calls and reads that become too short after trimming.

Any idea why we would want to do this before aligning to the reference genome?

**Input your response to the question in the text box below:**

We need to trim the raw data to remove sequencing adapters and low-quality bases from the ends of the reads. This ensures that only high-quality, biologically relevant sequences are aligned to the reference genome, preventing errors in downstream analysis.

<br>

<div class="alert alert-block alert-success">

<details>
<summary><b>Answer</b></summary>

<br>

Removing any bases that were artificially added and thus were not generated from the organism’s DNA (represented by the reference genome), will improve the quality of alignment.

</details>
</div>

<br>
Let's see how this is done by trimming the forward (R1) and reverse (R2) reads of sample FLT_M23 with Trim Galore.

Before we can run Trim Galore, we first need to install it and Cutadapt, which is used by Trim Galore, by running the following commands:

In [20]:
# install Trim Galore
import os
if not os.path.exists(f"{GL4U_RNASEQ_DIR}/TrimGalore/TrimGalore-0.6.10/trim_galore"):
  !mkdir {GL4U_RNASEQ_DIR}/TrimGalore
  !wget -O {GL4U_RNASEQ_DIR}/TrimGalore/trim_galore.tar.gz https://github.com/FelixKrueger/TrimGalore/archive/0.6.10.tar.gz
  !curl -fsSL https://github.com/FelixKrueger/TrimGalore/archive/0.6.10.tar.gz -o {GL4U_RNASEQ_DIR}/TrimGalore/trim_galore.tar.gz
  !tar xzf {GL4U_RNASEQ_DIR}/TrimGalore/trim_galore.tar.gz -C {GL4U_RNASEQ_DIR}/TrimGalore
# make the trim_galore command executable
!chmod +x {GL4U_RNASEQ_DIR}/TrimGalore/TrimGalore-0.6.10/trim_galore

In [21]:
# check trim_galore version
!{GL4U_RNASEQ_DIR}/TrimGalore/TrimGalore-0.6.10/trim_galore -v


                        Quality-/Adapter-/RRBS-/Speciality-Trimming
                                [powered by Cutadapt]
                                  version 0.6.10

                               Last update: 02 02 2023



In [22]:
# install cutadapt
!pip install cutadapt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.1/281.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.1/106.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 11.4 MB/s eta 0:00:00


In [23]:
# find path to cutadapt executable
!which cutadapt

/usr/local/bin/cutadapt


We're now ready to trim the forward (R1) and reverse (R2) reads of sample FLT_M23 with Trim Galore.


In [24]:
# run trim_galore on R1 and R2
!{GL4U_RNASEQ_DIR}/TrimGalore/TrimGalore-0.6.10/trim_galore \
 --path_to_cutadapt /usr/local/bin/cutadapt \
 --gzip \
 --cores 2 \
 --phred33 \
 --paired \
 --output_dir $trimmed_fastq \
 $raw_fastq/FLT_M23_R1_raw.fastq.gz $raw_fastq/FLT_M23_R2_raw.fastq.gz

Path to Cutadapt set as: '/usr/local/bin/cutadapt' (user defined)
Cutadapt seems to be working fine (tested command '/usr/local/bin/cutadapt --version')
Cutadapt version: 5.2
Cutadapt seems to be using Python 3! Proceeding with multi-core enabled Cutadapt using 2 cores
pigz 2.6
Parallel gzip (pigz) detected. Proceeding with multicore (de)compression using 2 cores

Proceeding with 'pigz -p 2' for decompression
To decrease CPU usage of decompression, please install 'igzip' and run again

Output will be written into the directory: /content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/01-TG_Preproc/Fastq/


AUTO-DETECTING ADAPTER TYPE
Attempting to auto-detect adapter type from the first 1 million sequences of the first file (>> /content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/00-RawData/Fastq/FLT_M23_R1_raw.fastq.gz <<)

Found perfect matches for the following adapter sequences:
Adapter type	Count	Sequence	Sequences analysed	Percentage
Illumina	1737	AGATCGGAAGAGC	100000	1.74
smallRNA	1	TGGAATTCTCGG	

The suffixes added to the trimmed fastq output files by Trim Galore are not very intuitive so the GeneLab pipeline renames them as follows:

In [25]:
!mv $trimmed_fastq/FLT_M23_R1_raw_val_1.fq.gz $trimmed_fastq/FLT_M23_R1_trimmed.fastq.gz
!mv $trimmed_fastq/FLT_M23_R2_raw_val_2.fq.gz $trimmed_fastq/FLT_M23_R2_trimmed.fastq.gz

<div class="alert alert-block alert-info">
<b><i>Code Breakdown</i></b>

**Parameter Definitions:**

* `--path-to-cutadapt` – specifies the path to the cutadapt executable
* `--gzip` – compress the output files with `gzip`
* `--cores` – specifies the number of compute cores to use
* `--phred33` – instructs cutadapt to use ASCII+33 quality scores as Phred scores for quality trimming
* `--paired` – indicates paired-end reads - both reads, forward (R1) and reverse (R2) must pass length threshold or else both reads are removed (if not defined, a 20bp length threshold is applied)
* `--output_dir` – the output directory to store the trimmed fastq and respective trimming report files
* `$raw_fastq/${sample}_R1_raw.fastq.gz $raw_fastq/${sample}_R2_raw.fastq.gz` – the input reads are specified as a positional argument, paired-end read files are listed pairwise such that the forward reads (\*R1_raw.fastq.gz) are immediately followed by the respective reverse reads (\*R2_raw.fastq.gz) for each sample

**Input Data:**
- `*raw.fastq.gz` (raw reads)

**Output Data:**
- `*trimming_report.txt` (trimming report: information about how the data were trimmed/filtered)
- `*trimmed.fastq.gz` (trimmed reads: remaining sequence data after quality, adapter, and read length filtering)

</div>

<br>___________

<br>
Let's list the output files that were generated:

In [26]:
!ls -1 $trimmed_fastq/

FLT_M23_R1_raw.fastq.gz_trimming_report.txt
FLT_M23_R1_trimmed.fastq.gz
FLT_M23_R2_raw.fastq.gz_trimming_report.txt
FLT_M23_R2_trimmed.fastq.gz


Let's view the trimming reports for both the forward (R1) and reverse (R2) read data from sample FLT_M23:

In [27]:
!echo "Forward trimming report:"
!cat $trimmed_fastq/FLT_M23_R1_raw.fastq.gz_trimming_report.txt
!echo ""
!echo ""
!echo "Reverse trimming report:"
!cat $trimmed_fastq/FLT_M23_R2_raw.fastq.gz_trimming_report.txt

Forward trimming report:

SUMMARISING RUN PARAMETERS
Input filename: /content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/00-RawData/Fastq/FLT_M23_R1_raw.fastq.gz
Trimming mode: paired-end
Trim Galore version: 0.6.10
Cutadapt version: 5.2
Python version: 3.12.13
Number of cores used for trimming: 2
Quality Phred score cutoff: 20
Quality encoding type selected: ASCII+33
Using Illumina adapter for trimming (count: 1737). Second best hit was smallRNA (count: 1)
Adapter sequence: 'AGATCGGAAGAGC' (Illumina TruSeq, Sanger iPCR; auto-detected)
Maximum trimming error rate: 0.1 (default)
Minimum required adapter overlap (stringency): 1 bp
Minimum required sequence length for both reads before a sequence pair gets removed: 20 bp
Output file will be GZIP compressed


This is cutadapt 5.2 with Python 3.12.13
Command line parameters: -j 2 -e 0.1 -q 20 -O 1 -a AGATCGGAAGAGC /content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/00-RawData/Fastq/FLT_M23_R1_raw.fastq.gz
Processing single-end reads on 2 cores ...

===

---

**Take a look at the === Summary === section of both the forward and reverse read trimming reports and answer the following questions:**

1. How many forward reads were processed? How many reverse reads?

2. Are the total number of forward and reverse reads processed the same or different? Should these numbers be the same? Why?

3. Were any adapters detected in the forward or reverse reads? If yes, how many forward reads contained adapters? How many reverse reads contained adapters?

4. After adapters were trimmed from the forward and reverse reads, were any reads removed? Why?

5. How many basepairs were removed due to poor quality in the forward reads? What about the reverse reads?

6. After adapter trimming and quality filtering, what is the total number and percent of basepairs remaining for the forward reads? What about the reverse reads?

7. Did the raw forward or raw reverse reads have better quality? How do you know?

**Look at the bottom of the reverse read trimming report and answer the following questions:**

8. After trimming and filtering, did any read pairs fail to meet the 20bp read length cutoff? If yes, how many?

9. Why is this information only contained in the reverse read trimming report?

**Input your responses to each question in the text boxes below:**
> *Answer each question in your own words BEFORE checking your answers against the solutions below. Please do not copy and paste the answers provided.*


Question 1: In this pre-processing step, 100,000 reads were processed for both forward and reverse directions.


Question 2: In a paired-end sequencing library, these counts must be identical because each unique cDNA fragment is sequenced from both the 5' and 3' ends, generating a pair of reads for every single molecule.


Question 3: Yes, significant adapter contamination was identified. Approximately 31,739 of forward reads and 32,132 of reverse reads contained Illumina universal adapters. This high percentage of non-biological sequences necessitated immediate trimming to prevent mismatches during genomic alignment.


Question 4: No reads were entirely discarded at this specific sub-stage.


Question 5: In the forward reads, 7,848 bp were removed due to poor base quality. For the reverse reads, a significantly higher amount, 49,988 bp, was removed.


Question 6: After processing, the forward reads have 9,893,689 bp remaining. The reverse reads have 9,849,322 bp remaining.


Question 7: The forward reads exhibited better overall quality. This is evidenced by the lower percentage of basepairs removed due to low-quality scores and the higher total amount of high-quality basepairs remaining after the filtering process.


Question 8: Yes, 89 read pairs failed to meet the minimum 20bp length requirement after trimming. These specific sequences were discarded to ensure that only reads with enough information for unique genomic mapping were kept for the next stage.


Question 9: This is because, in paired-end sequencing, the final decision to reject a fragment is made during the validation of the reverse read. If a reverse read falls below the length threshold, the entire pair is removed to maintain data synchronization.


<br>

<div class="alert alert-block alert-success">

<details>
<summary><b>Q1 Solution</b></summary>

<br>

Number of forward reads processed: 100,000
Number of reverse reads processed: 100,000

</details>
</div>


<div class="alert alert-block alert-success">

<details>
<summary><b>Q2 Solution</b></summary>

<br>

The total number of forward and reverse reads processed are the same.

They should be the same because the same cDNA fragment was sequenced in the forward and reverse directions.

</details>
</div>


<div class="alert alert-block alert-success">

<details>
<summary><b>Q3 Solution</b></summary>

<br>

Yes, adapters were detected in both the forward and reverse reads.
Number of forward reads with adapters: 31,739 (31.7%)
Number of reverse reads with adapters: 32,132 (32.1%)

</details>
</div>


<div class="alert alert-block alert-success">

<details>
<summary><b>Q4 Solution</b></summary>

<br>

No reads were removed after adapter trimming because after removing basepairs due to adapter contamination, all forward and reverse reads still contained basepairs.

</details>
</div>


<div class="alert alert-block alert-success">

<details>
<summary><b>Q5 Solution</b></summary>

<br>

Basepairs removed due to poor quality in the forward reads: 7,848 bp (0.1%)
Basepairs removed due to poor quality in the reverse reads: 49,988 bp (0.5%)
    
</details>
</div>


<div class="alert alert-block alert-success">

<details>
<summary><b>Q6 Solution</b></summary>

<br>

Total amount of basepairs remaining in the forward reads: 9,893,689 bp (98.9%)
Total amount of basepairs remaining in the reverse reads: 9,849,322 bp (98.5%)
    
</details>
</div>


<div class="alert alert-block alert-success">

<details>
<summary><b>Q7 Solution</b></summary>

<br>

Forward reads had better quality because fewer base pairs were removed due to adapter contamination and/or poor quality.
> Note: Read quality can also be determined by the multiqc report of the raw fastqc data as discussed above.

</details>
</div>


<div class="alert alert-block alert-success">

<details>
<summary><b>Q8 Solution</b></summary>

<br>

Yes, 89 (0.09%) of read pairs failed to meet the 20bp read length cutoff.

</details>
</div>

<div class="alert alert-block alert-success">

<details>
<summary><b>Q9 Solution</b></summary>

<br>

This information is only contained in the reverse read trimming report because the number of forward and reverse reads have to be the same. So, the if the forward read of a fragment was 40bp after trimming (passing the 20bp cutoff) but the respective reverse read of that same fragment was 19bp after trimming (failing the 20bp cutoff) then both the forward and reverse read for the fragment must be removed.

</details>
</div>


<br>


<div class="alert alert-block alert-info">
<b>Note:</b><br>
    
To preserve compute resources, raw sequence data for all samples have been trimmed with these parameters and the resulting trimmed reads will be used for the following steps.

</div>

---

<a class="anchor" id="trimmedfastqc"></a>

### 2b. Trimmed Data QC with FastQC

Now that we've trimmed/filtered our raw sequence data, we will assess the quality of the trimmed sequence data using [FastQC](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/) to evaluate how effective trimming was at improving sequence quality.

Let's run FastQC on the trimmed reads of our FLT_M23 sample using the following command:

In [28]:
!{GL4U_RNASEQ_DIR}/FastQC/fastqc -o $trimmed_fastqc $trimmed_fastq/*trimmed.fastq.gz

application/gzip
Started analysis of FLT_M23_R1_trimmed.fastq.gz
application/gzip
Approx 5% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 10% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 15% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 20% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 25% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 30% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 35% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 40% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 45% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 50% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 55% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 60% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 65% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 70% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 75% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 80% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 85% complete for FLT_M23_R1_trimmed.fastq.gz
Approx 90% complete for FLT_M23_R1_

<div class="alert alert-block alert-info">
<b><i>Code Breakdown</i></b>

**Parameter Definitions:**

* `-o` – the output directory to store the FastQC results
* `*trimmed.fastq.gz` – the input reads are specified as a positional argument, and can be given all at once with wildcards, or as individual arguments with spaces in between them

**Input Data:**
- `*trimmed.fastq.gz` (trimmed reads)

**Output Data:**
- `*fastqc.html` (FastQC report: a webpage file containing graphical representations of the trimmed sequence quality)
- `*fastqc.zip` (FastQC data: a compressed directory containing the quality reports of all trimmed sequences assessed)

</div>

<br>
---

<br>Let's list the FastQC output files that were generated:

In [29]:
!ls -1 $trimmed_fastqc/FLT_M23*

/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/01-TG_Preproc/FastQC_Reports/FLT_M23_R1_trimmed_fastqc.html
/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/01-TG_Preproc/FastQC_Reports/FLT_M23_R1_trimmed_fastqc.zip
/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/01-TG_Preproc/FastQC_Reports/FLT_M23_R2_trimmed_fastqc.html
/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/01-TG_Preproc/FastQC_Reports/FLT_M23_R2_trimmed_fastqc.zip


Open the trimmed data `FastQC_Reports` directory in the side panel of this Colab notebook by clicking: `/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/01-TG_Preproc/FastQC_Reports`. Next, click the 3 dots next to the `FLT_M23_R1_trimmed_fastqc.html` and `FLT_M23_R2_trimmed_fastqc.html` files in the left side panel to download them to your computer. Then navigate to the HTML files you downloaded on your computer and double click on them to open in your web browser to view the FastQC reports.

<div class="alert alert-block alert-info">
<b>Note:</b><br>
    
To preserve compute resources, the FastQC data and reports have been generated for the forward (R1) and reverse (R2) trimmed fastq files of all OSD-104 FLT and GC samples.

</div>

---

<a class="anchor" id="trimmedmultiqc"></a>

### 2c. Compile Trimmed Data QC with MultiQC

Rather than viewing the FastQC reports of each trimmed fastq file for all 12 samples individually, we'll again use the [MultiQC](https://docs.seqera.io/multiqc) program but this time to compile the trimmed FastQC data from all samples and generate a single report (and associated data files). This will allow us to view the trimmed sequence data quality of all samples at once and compare it to the raw sequence data quality.

Let's compile the trimmed FastQC data from our 12 samples using MultiQC by running the following command:

In [30]:
!multiqc --interactive -n trimmed_multiqc -o $trimmed_multiqc $trimmed_fastqc/


/// ]8;id=920498;https://multiqc.info\MultiQC]8;;\ 🔍 v1.33

       file_search | Search path: /content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/01-TG_Preproc/FastQC_Reports
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 26/26  
            fastqc | Found 24 reports
     write_results | Data        : OSD-104/01-TG_Preproc/FastQC_Reports/trimmed_multiqc_report/trimmed_multiqc_data
     write_results | Report      : OSD-104/01-TG_Preproc/FastQC_Reports/trimmed_multiqc_report/trimmed_multiqc.html
           multiqc | MultiQC complete


<div class="alert alert-block alert-info">
<b><i>Code Breakdown</i></b>

**Parameter Definitions:**

* `--interactive` – force reports to use interactive plots
* `-n` – prefix name for the output files
* `-o` – the output directory to store the MultiQC results
* `$trimmed_fastqc/` – the directory holding the trimmed FastQC data from each sample, provided as a positional argument

**Input Data:**
- `*fastqc.zip` (trimmed FastQC data)

**Output Data:**
- `trimmed_multiqc_data` (directory containing the compiled trimmed FastQC data from each sample, used to create the MultiQC report)
- `trimmed_multiqc.html` (MultiQC report: an interactive webpage file containing graphical representations of the compiled trimmed sequence quality)

</div>



---
MultiQC is finished running when the loading animation on the left end of the code cell stops and a green checkmark appears. In a standard bash shell environment, if the multiQC job finishes successfully, you will see a "MultiQC complete" statement at the end of the standard output, and could check that the number of reports found matches what you expect.
> Note: For paired-end data, we expect a report for both the forward (R1) and reverse (R2) reads of each sample. Since we have 12 paired-end samples, MultiQC should detect 24 FastQC reports.

Let's list the output files that were generated:

In [31]:
!ls -1 $trimmed_multiqc/*

/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/01-TG_Preproc/FastQC_Reports/trimmed_multiqc_report/trimmed_multiqc.html

/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/01-TG_Preproc/FastQC_Reports/trimmed_multiqc_report/trimmed_multiqc_data:
fastqc_adapter_content_plot.txt
fastqc_overrepresented_sequences_plot.txt
fastqc_per_base_n_content_plot.txt
fastqc_per_base_sequence_quality_plot.txt
fastqc_per_sequence_gc_content_plot_Counts.txt
fastqc_per_sequence_gc_content_plot_Percentages.txt
fastqc_per_sequence_quality_scores_plot.txt
fastqc_sequence_counts_plot.txt
fastqc_sequence_duplication_levels_plot.txt
fastqc_sequence_length_distribution_plot.txt
fastqc-status-check-heatmap.txt
fastqc_top_overrepresented_sequences_table.txt
llms-full.txt
multiqc_citations.txt
multiqc_data.json
multiqc_fastqc.txt
multiqc_general_stats.txt
multiqc.log
multiqc.parquet
multiqc_software_versions.txt
multiqc_sources.txt


Open the `trimmed_multiqc_report` directory in the side panel of this Colab notebook by clicking: `/content/mnt/MyDrive/NASA/GL4U/RNAseq/OSD-104/01-TG_Preproc/FastQC_Reports/trimmed_multiqc_report`, as shown below.
> *Note: These files are stored in your Google Drive folder.*

<br>

<div align="center">
<img src="https://raw.githubusercontent.com/nasa/GeneLab-Training/refs/heads/GL4U_RNAseq_2024_Colab/images/gd_trim_multiqc_dir.png" alt="Open the trimmed multiqc report directory">
</div>
<br>

Click the 3 dots next to the `trimmed_multiqc.html` file in the left side panel to download it to your computer as shown below. Then navigate to the HTML file on your computer and double click on it to open in your web browser to view the compiled FastQC reports for all 12 samples.

<br>

<div align="center">
<img src="https://raw.githubusercontent.com/nasa/GeneLab-Training/refs/heads/GL4U_RNAseq_2024_Colab/images/gd_dl_trim_multiqc.png" alt="Download trimmed multiQC">
</div>
<br>

Navigate to the HTML file you downloaded on your computer then double click on it to open in your web browser to view the compiled trimmed FastQC reports for all 12 samples, which should look similar to the following:

<br>

<div align="center">
<img src="https://raw.githubusercontent.com/nasa/GeneLab-Training/refs/heads/GL4U_RNAseq_2024_Colab/images/gd_open_trim_multiqc.png" alt="Open the trimmed multiqc report file">
</div>
<br>
<div class="alert alert-block alert-info">
<b>Note:</b><br>
    
Click [here](https://docs.seqera.io/multiqc/modules/fastqc) for a description of the MultiQC plots derived from FastQC data.

</div>

<div class="alert alert-block alert-warning">

Be sure to click the "Trust HTML" button in the top left corner of the MultiQC report to be able to see all the plots.

If the plots are still not visible, make sure JavaScript is enabled on the web-browser you are using. See [above](#Enable-JavaScript) for instructions to enable JavaScript on common web-browsers.

</div>

**Take a look at the multiQC report of the trimmed fastQC data above and answer the following questions:**

1. How many reads are there in the GC_M33 sample after trimming? What about the FLT_M24 sample?  (Hint: See *M Seqs* under "General Statistics")

2. What is the sequencing depth range among samples?  

3. What do you notice about the quality of the trimmed sequence data compared with the raw?

4. Do you still see adapters detected?

5. Do you think we're ready to align the trimmed reads to the reference genome now? Why or why not?

**Input your responses to each question in the text boxes below:**
> *Answer each question in your own words BEFORE checking your answers against the solutions below. Please do not copy and paste the answers provided.*


Question 1: After the trimming and filtering process, the GC_M33 sample retains 99,858 reads, while the FLT_M24 sample has 99,879 reads.


Question 2: The sequencing depth remains highly consistent across all samples, ranging from approximately 99,850 for FLT_M26 to 99,916 for FLT_M28 reads.


Question 3: There is a significant improvement in the overall sequence quality. By removing low-quality bases from the ends, we have shifted the mean quality score into the "green zone," which is optimal for accurate alignment.


Question 4: No, adapters are no longer detected.

Question 5: Yes, we are now fully prepared for genomic alignment. The pre-processing steps have yielded high-quality, clean reads free from adapter contamination and sequencing artifacts. Using this refined dataset will maximize the mapping rate and ensure that the resulting alignment is precise.

<br>

<div class="alert alert-block alert-success">

<details>
<summary><b>Q1 Solution</b></summary>

<br>

Number of reads in GC_M33 after trimming: 99,858
Number of reads in FLT_M24 after trimming: 99,879
> _Note: Since this is a truncated dataset and the "M Seqs" column is rounded to the neareast tenths place, all samples in this dataset will appear to have the same read depth (0.1M). However, you can add up the number of unique + dupe sequences in the "Sequence Counts" section for each sample to get these exact values. Alternatively, you can look at the individual fastqc HTML reports for each sample in the `$trimmed_fastqc` location._
    
</details>
</div>


<div class="alert alert-block alert-success">

<details>
<summary><b>Q2 Solution</b></summary>

<br>

99,850 (FLT_M26) - 99,916 (FLT_M28)
    
</details>
</div>


<div class="alert alert-block alert-success">

<details>
<summary><b>Q3 Solution</b></summary>

<br>

The trimmed sequence data is better quality than the raw data.

</details>
</div>


<div class="alert alert-block alert-success">

<details>
<summary><b>Q4 Solution</b></summary>

<br>

No, adapters are no longer detected.

</details>
</div>


<div class="alert alert-block alert-success">

<details>
<summary><b>Q5 Solution</b></summary>

<br>

Yes, because the trimmed reads are high quality and do not contain any detectable adapters.

</details>
</div>

---

We have now successfully pre-processed the sequence data for all 12 samples in our study (OSD-104). We will use the trimmed reads, stored in the `*trimmed.fastq.gz` files, in the next notebook for alignment then quantitation.



<a class="anchor" id="copy-to-gd"></a>

## 3. Copy Completed JN to Google Drive
Double check all sections in this JN to make sure that all activities have been completed. Then, save a copy of your completed JN to your Google Drive by selecting "Save a copy in Drive" under "File" in the top left corner of this JN as shown below.

<br>

<div align="center">
<img src="https://raw.githubusercontent.com/nasa/GeneLab-Training/refs/heads/GL4U_RNAseq_2024_Colab/images/save_RNAseq_01_JN.png" alt="Save JN">
</div>

<br>
After successfully saving your completed JN to your Google Drive, open the <a href="https://github.com/nasa/GeneLab-Training/blob/GL4U_RNAseq_2024_Colab/On-Demand/Access_Processing_1of2_JN.md/#upload-your-completed-jn-to-canvas">Access_Processing_1of2_JN</a> instructions in a new tab, and follow the steps in the "Upload Your Completed JN To Canvas" section to download your completed JN then upload it to Canvas to receive credit.
<br>